# Проект. Нейросеть для предсказания калорийности блюд

**Автор:** avo_m25_555

**Задача:** по фотографии блюда и списку ингредиентов предсказать калорийность (ккал).

**Целевая метрика:** MAE &lt; 50 на test.

Ноутбук **монолитный**: весь код этапов 1–4 внутри ячеек, без внешних `.py`-модулей проекта.


## Чек-лист ТЗ

| Этап | Содержание |
|------|------------|
| 1 | EDA, merge id→названия, видение решения и метрик |
| 2 | Dataset / transforms / модель / `train(cfg)` / seed |
| 3 | Регрессия (head→1, L1Loss, BATCH=4), обучение, чекпоинт |
| 4 | Инференс на test, MAE, топ-5 ошибок с разбором |


## Этап 0. Окружение и путь к `nutrition/`


In [ ]:
# @title 0.1. Зависимости (Colab)
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "timm",
            "albumentations",
            "transformers",
            "torchmetrics",
            "pillow",
            "pandas",
            "tqdm",
            "matplotlib",
        ]
    )
print("IN_COLAB =", IN_COLAB)


In [ ]:
# @title 0.2. Импорты, seed, DATA_ROOT (= папка nutrition/)
from __future__ import annotations

import copy
import os
import random
import re
import shutil
import sys
import time
import zipfile
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import albumentations as A
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torchmetrics
from albumentations.pytorch import ToTensorV2
from PIL import Image
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

IN_COLAB = "google.colab" in sys.modules
DATASET_DIRNAME = "nutrition"
DATASET_ZIPNAME = "nutrition.zip"


def _is_nutrition_root(p: Path) -> bool:
    """Корень датасета: dish.csv + ingredients.csv + images/."""
    return (
        p.is_dir()
        and (p / "dish.csv").is_file()
        and (p / "ingredients.csv").is_file()
        and (p / "images").is_dir()
    )


def _notebook_dir() -> Path:
    """Папка ноутбука (обычно cwd) или поиск avo_Nutrition_rep вверх по дереву."""
    cwd = Path.cwd().resolve()
    if (cwd / "avo_m25_555_nutrition.ipynb").is_file() or _is_nutrition_root(cwd / DATASET_DIRNAME):
        return cwd
    for base in [cwd, *cwd.parents]:
        if (base / "avo_m25_555_nutrition.ipynb").is_file():
            return base
        nested = base / "avo_Nutrition_rep"
        if (nested / "avo_m25_555_nutrition.ipynb").is_file():
            return nested
    return cwd


def _find_nutrition_near(notebook_dir: Path) -> Path | None:
    """Сначала nutrition/ (или nutrition.zip → распаковка) рядом с ноутбуком."""
    for c in (
        notebook_dir / DATASET_DIRNAME,
        notebook_dir / "avo_Nutrition_rep" / DATASET_DIRNAME,
        Path("/content") / DATASET_DIRNAME,
        Path("/content/avo_Nutrition_rep") / DATASET_DIRNAME,
    ):
        if _is_nutrition_root(c):
            return c
    return None


def _extract_zip(zpath: Path, dest: Path) -> Path | None:
    if not zpath.is_file():
        return None
    with zpath.open("rb") as f:
        if f.read(2) != b"PK":
            print("Файл не ZIP:", zpath)
            return None
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(dest)
    if _is_nutrition_root(dest):
        return dest
    for child in dest.iterdir():
        if child.is_dir() and _is_nutrition_root(child):
            return child
    for dish in dest.glob("*/dish.csv"):
        if _is_nutrition_root(dish.parent):
            return dish.parent
    return None


def _pick_local_folder() -> Path | None:
    """Диалог выбора папки датасета (nutrition/) с локального диска."""
    try:
        import tkinter as tk
        from tkinter import filedialog
    except Exception as e:
        print("Диалог папки недоступен:", e)
        return None
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    chosen = filedialog.askdirectory(
        title="Выберите папку nutrition (dish.csv + ingredients.csv + images/)"
    )
    root.destroy()
    return Path(chosen) if chosen else None


def _upload_zip_colab(save_as: Path) -> Path | None:
    from google.colab import files  # type: ignore

    print("Загрузите nutrition.zip (Choose Files)…")
    uploaded = files.upload()
    if not uploaded:
        return None
    name, raw = next(iter(uploaded.items()))
    save_as.parent.mkdir(parents=True, exist_ok=True)
    target = save_as if str(save_as).lower().endswith(".zip") else save_as / DATASET_ZIPNAME
    Path(target).write_bytes(raw)
    print("Сохранено:", target, "из", name)
    return Path(target)


NOTEBOOK_DIR = _notebook_dir()
print("NOTEBOOK_DIR =", NOTEBOOK_DIR)

DATA_ROOT = _find_nutrition_near(NOTEBOOK_DIR)

if DATA_ROOT is None:
    for z in (
        NOTEBOOK_DIR / DATASET_ZIPNAME,
        NOTEBOOK_DIR / "avo_Nutrition_rep" / DATASET_ZIPNAME,
        Path("/content") / DATASET_ZIPNAME,
    ):
        if z.is_file():
            print("Найден zip:", z)
            DATA_ROOT = _extract_zip(z, NOTEBOOK_DIR / DATASET_DIRNAME)
            if DATA_ROOT is not None:
                break

if DATA_ROOT is None:
    print("nutrition/ рядом с ноутбуком не найдена — выберите папку на диске или загрузите nutrition.zip.")
    if IN_COLAB:
        z = _upload_zip_colab(Path("/content") / DATASET_ZIPNAME)
        if z is not None:
            DATA_ROOT = _extract_zip(z, Path("/content") / DATASET_DIRNAME)
    else:
        picked = _pick_local_folder()
        if picked is not None:
            print("Выбрано:", picked)
            if _is_nutrition_root(picked):
                DATA_ROOT = picked
            elif _is_nutrition_root(picked / DATASET_DIRNAME):
                DATA_ROOT = picked / DATASET_DIRNAME
            else:
                for z in list(picked.glob(DATASET_ZIPNAME)) + list(picked.glob("*.zip")):
                    DATA_ROOT = _extract_zip(z, NOTEBOOK_DIR / DATASET_DIRNAME)
                    if DATA_ROOT is not None:
                        break

assert DATA_ROOT is not None and _is_nutrition_root(DATA_ROOT), (
    "Датасет не найден. Нужна папка nutrition/ рядом с ноутбуком "
    "(dish.csv, ingredients.csv, images/) либо файл nutrition.zip / выбор папки / Upload."
)

ARTIFACTS = Path("/content/nutrition_artifacts") if IN_COLAB else (NOTEBOOK_DIR / "artifacts")
ARTIFACTS.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = str(ARTIFACTS / "best.pt")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DATA_ROOT (nutrition) =", DATA_ROOT)
print("CHECKPOINT_PATH =", CHECKPOINT_PATH)
print("DEVICE =", DEVICE)
if DEVICE.type == "cuda":
    print(torch.cuda.get_device_name(0))


## Этап 1. Исследовательский анализ (EDA)


In [ ]:
# @title 1.1. Загрузка таблиц и merge id → названия ингредиентов
def build_ingredient_map(ingredients_csv: Path) -> dict[int, str]:
    df = pd.read_csv(ingredients_csv)
    return {int(r["id"]): str(r["ingr"]).strip() for _, r in df.iterrows()}


def parse_ingredient_ids(raw: str) -> list[int]:
    if not isinstance(raw, str) or not raw.strip():
        return []
    ids = []
    for part in raw.split(";"):
        m = re.search(r"(\d+)$", part.strip())
        if m:
            ids.append(int(m.group(1)))
    return ids


def ingredients_to_text(raw: str, id2name: dict[int, str]) -> str:
    names = [id2name[i] for i in parse_ingredient_ids(raw) if i in id2name]
    return ", ".join(names) if names else "unknown"


def load_dish_table(data_root: Path) -> pd.DataFrame:
    dish = pd.read_csv(data_root / "dish.csv")
    id2name = build_ingredient_map(data_root / "ingredients.csv")
    dish = dish.copy()
    dish["ingredients_text"] = dish["ingredients"].map(lambda x: ingredients_to_text(x, id2name))
    dish["image_path"] = dish["dish_id"].map(
        lambda d: str(data_root / "images" / str(d) / "rgb.png")
    )
    dish["exists"] = dish["image_path"].map(os.path.isfile)
    return dish


dish = load_dish_table(DATA_ROOT)
print("shape:", dish.shape)
print("columns:", dish.columns.tolist())
print("\nsplit:\n", dish["split"].value_counts().to_string())
print("\nmissing images:", int((~dish["exists"]).sum()))
print("\nпример merge (как в ТЗ):")
raw0 = dish.iloc[0]["ingredients"]
print("  raw ids:", raw0[:80], "...")
print("  text   :", dish.iloc[0]["ingredients_text"][:120])
print("\nhead:")
print(
    dish[["dish_id", "split", "total_calories", "total_mass", "ingredients_text"]]
    .head(5)
    .to_string(index=False)
)


In [ ]:
# @title 1.2. Распределения калорий / массы / числа ингредиентов
dish["n_ingr"] = dish["ingredients_text"].str.split(",").str.len()

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, col, title in zip(
    axes,
    ["total_calories", "total_mass", "n_ingr"],
    ["Калории (ккал)", "Масса", "Число ингредиентов"],
):
    for split, color in [("train", "C0"), ("test", "C1")]:
        vals = dish.loc[dish["split"] == split, col]
        ax.hist(vals, bins=40, alpha=0.55, label=split, color=color)
    ax.set_title(title)
    ax.legend()
plt.tight_layout()
plt.show()

print(dish.groupby("split")[["total_calories", "total_mass", "n_ingr"]].describe().round(1))


In [ ]:
# @title 1.3. Визуализация примеров (фото + текст + калории)
train_ok = dish[dish["exists"] & (dish["split"] == "train")]
sample = train_ok.sample(n=min(4, len(train_ok)), random_state=SEED)

fig, axes = plt.subplots(1, min(4, len(sample)), figsize=(14, 3.5))
if min(4, len(sample)) == 1:
    axes = [axes]
for ax, (_, row) in zip(axes, sample.iterrows()):
    ax.imshow(Image.open(row["image_path"]).convert("RGB"))
    ax.set_title(
        f"{row['total_calories']:.0f} kcal\n{row['ingredients_text'][:40]}…",
        fontsize=8,
    )
    ax.axis("off")
plt.suptitle("Примеры train")
plt.tight_layout()
plt.show()


### Вывод этапа 1 — видение решения

**Как решаем задачу.** Мультимодальная **регрессия**: энкодер изображения (EfficientNet-B0) + энкодер текста ингредиентов (DistilBERT) → проекции → `Linear → 1` (ккал). Вход: `rgb.png` и строка названий продуктов после merge id→`ingr`.

**Данные.** Train/test заданы колонкой `split`. Из train отложим небольшую val (доля в конфиге) для early stopping. Технические `ingr_…` заменены на читаемые названия — иначе текстовая ветка бесполезна.

**Аугментации.** В выбранном конфиге (после CPU-поиска) слабые/отключённые aug: витринные фото блюд легко «ломаются» сильным ColorJitter/геометрией; важнее стабильный Resize + ImageNet-нормализация. При необходимости — лёгкий HorizontalFlip.

**Метрика.** Основная — **MAE (ккал)** на test; цель продукта MAE &lt; 50. На обучении тот же смысл через `nn.L1Loss`. Дополнительно смотрим val MAE для выбора чекпоинта.


## Этап 2. Пайплайн обучения (код в ноутбуке)

Ниже — Dataset, transforms, модель, `train(cfg)` и seed. Логика соответствует требованиям ТЗ к `dataset.py` / `utils.py`, но **собрана в ноутбуке** (монолит).


In [ ]:
# @title 2.1. Transforms, Dataset, DataLoader
def get_train_transforms(image_size: int, use_aug: bool = True) -> A.Compose:
    ops = [A.Resize(image_size, image_size)]
    if use_aug:
        ops.extend(
            [
                A.HorizontalFlip(p=0.5),
                A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, p=0.5),
            ]
        )
    ops.extend(
        [
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ]
    )
    return A.Compose(ops)


def get_eval_transforms(image_size: int) -> A.Compose:
    return A.Compose(
        [
            A.Resize(image_size, image_size),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ]
    )


class CaloriesDataset(Dataset):
    """image (rgb.png) + ingredients text → calories."""

    def __init__(self, frame: pd.DataFrame, transform: A.Compose, tokenizer=None, max_length: int = 64):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int) -> dict:
        row = self.frame.iloc[idx]
        # путь: nutrition/images/<dish_id>/rgb.png
        image = Image.open(row["image_path"]).convert("RGB")
        image_t = self.transform(image=np.array(image))["image"]
        text = str(row["ingredients_text"])
        item = {
            "image": image_t,
            "text": text,
            "calories": float(row["total_calories"]),
            "dish_id": str(row["dish_id"]),
            "mass": float(row["total_mass"]),
        }
        if self.tokenizer is not None:
            enc = self.tokenizer(
                text,
                padding="max_length",
                truncation=True,
                max_length=self.max_length,
                return_tensors="pt",
            )
            item["input_ids"] = enc["input_ids"].squeeze(0)
            item["attention_mask"] = enc["attention_mask"].squeeze(0)
        return item


def collate_calories(batch: list[dict]) -> dict:
    out = {
        "image": torch.stack([b["image"] for b in batch], dim=0),
        "calories": torch.tensor([b["calories"] for b in batch], dtype=torch.float32),
        "text": [b["text"] for b in batch],
        "dish_id": [b["dish_id"] for b in batch],
        "mass": torch.tensor([b["mass"] for b in batch], dtype=torch.float32),
    }
    if "input_ids" in batch[0]:
        out["input_ids"] = torch.stack([b["input_ids"] for b in batch], dim=0)
        out["attention_mask"] = torch.stack([b["attention_mask"] for b in batch], dim=0)
    return out


def make_loaders(train_df, val_df, test_df, tokenizer, image_size, batch_size, use_aug, max_length=64, num_workers=0):
    train_ds = CaloriesDataset(train_df, get_train_transforms(image_size, use_aug), tokenizer, max_length)
    val_ds = CaloriesDataset(val_df, get_eval_transforms(image_size), tokenizer, max_length)
    test_ds = CaloriesDataset(test_df, get_eval_transforms(image_size), tokenizer, max_length)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, collate_fn=collate_calories)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, collate_fn=collate_calories)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, collate_fn=collate_calories)
    return train_loader, val_loader, test_loader


print("Dataset / transforms / loaders OK")


In [ ]:
# @title 2.2. Config, модель, train(), checkpoint
@dataclass
class Config:
    CONFIG: str = "full"
    SEED: int = 42
    DATA_ROOT: str = ""
    IMAGE_SIZE: int = 160
    BATCH_SIZE: int = 4
    EPOCHS: int = 12
    PATIENCE: int = 3
    HIDDEN_DIM: int = 128
    MAX_LENGTH: int = 64
    TEXT_MODEL: str = "distilbert-base-uncased"
    IMAGE_MODEL: str = "efficientnet_b0"
    TEXT_LR: float = 2e-5
    IMAGE_LR: float = 2e-5
    CLASSIFIER_LR: float = 1e-3
    WEIGHT_DECAY: float = 1e-4
    FREEZE_TEXT: bool = False
    FREEZE_IMAGE: bool = False
    USE_AUG: bool = False
    CHECKPOINT_PATH: str = "artifacts/best.pt"
    DEVICE: str = "auto"
    NUM_WORKERS: int = 0
    VAL_FRACTION: float = 0.1
    NAME: str = "full"


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class MultimodalCalorieModel(nn.Module):
    """Image + text → projections → Linear(hidden*2, 1)."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.image_encoder = timm.create_model(
            cfg.IMAGE_MODEL, pretrained=True, num_classes=0, global_pool="avg"
        )
        with torch.no_grad():
            img_dim = self.image_encoder(torch.zeros(1, 3, cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)).shape[-1]
        self.text_encoder = AutoModel.from_pretrained(cfg.TEXT_MODEL)
        text_dim = self.text_encoder.config.hidden_size
        self.image_projection = nn.Sequential(nn.Linear(img_dim, cfg.HIDDEN_DIM), nn.ReLU(), nn.Dropout(0.1))
        self.text_projection = nn.Sequential(nn.Linear(text_dim, cfg.HIDDEN_DIM), nn.ReLU(), nn.Dropout(0.1))
        self.head = nn.Linear(cfg.HIDDEN_DIM * 2, 1)  # ТЗ: один выход, без активации
        if cfg.FREEZE_IMAGE:
            for p in self.image_encoder.parameters():
                p.requires_grad = False
        if cfg.FREEZE_TEXT:
            for p in self.text_encoder.parameters():
                p.requires_grad = False

    def encode_text(self, input_ids, attention_mask):
        last = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        return (last * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)

    def forward(self, image, input_ids, attention_mask):
        img = self.image_projection(self.image_encoder(image))
        txt = self.text_projection(self.encode_text(input_ids, attention_mask))
        return self.head(torch.cat([img, txt], dim=-1)).squeeze(-1)


def build_optimizer(model: MultimodalCalorieModel, cfg: Config) -> AdamW:
    # ТЗ: все части модели с раздельными LR
    groups = [
        {"params": model.text_encoder.parameters(), "lr": cfg.TEXT_LR},
        {"params": model.image_encoder.parameters(), "lr": cfg.IMAGE_LR},
        {"params": model.text_projection.parameters(), "lr": cfg.CLASSIFIER_LR},
        {"params": model.image_projection.parameters(), "lr": cfg.CLASSIFIER_LR},
        {"params": model.head.parameters(), "lr": cfg.CLASSIFIER_LR},
    ]
    groups = [g for g in groups if any(p.requires_grad for p in g["params"])]
    return AdamW(groups, weight_decay=cfg.WEIGHT_DECAY)


@torch.no_grad()
def evaluate(model, loader, device) -> float:
    model.eval()
    metric = torchmetrics.MeanAbsoluteError().to(device)
    for batch in loader:
        pred = model(
            batch["image"].to(device),
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device),
        )
        metric.update(pred, batch["calories"].to(device))
    return float(metric.compute().item())


def train(cfg: Config, train_loader, val_loader, test_loader) -> dict[str, Any]:
    set_seed(cfg.SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") if cfg.DEVICE == "auto" else torch.device(cfg.DEVICE)
    cfg.DEVICE = str(device)
    model = MultimodalCalorieModel(cfg).to(device)
    opt = build_optimizer(model, cfg)
    criterion = nn.L1Loss()  # ТЗ: регрессия MAE
    history = {"train_mae": [], "val_mae": [], "epoch_sec": []}
    best_val, best_state, best_epoch, bad = float("inf"), None, 0, 0
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"=== Train {cfg.NAME} | trainable={n_params:,} | {device} ===")
    print(asdict(cfg))

    for epoch in range(1, cfg.EPOCHS + 1):
        model.train()
        t0 = time.perf_counter()
        run, n = 0.0, 0
        for batch in train_loader:
            image = batch["image"].to(device)
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            y = batch["calories"].to(device)
            opt.zero_grad(set_to_none=True)
            pred = model(image, ids, mask)
            loss = criterion(pred, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            run += torch.abs(pred.detach() - y).sum().item()
            n += y.numel()
        train_mae = run / max(1, n)
        val_mae = evaluate(model, val_loader, device)
        sec = time.perf_counter() - t0
        history["train_mae"].append(train_mae)
        history["val_mae"].append(val_mae)
        history["epoch_sec"].append(sec)
        mark = ""
        if val_mae < best_val - 1e-4:
            best_val, best_epoch, best_state, bad = val_mae, epoch, copy.deepcopy(model.state_dict()), 0
            mark = " *best*"
        else:
            bad += 1
        print(f"epoch {epoch}/{cfg.EPOCHS} | train_mae={train_mae:.2f} | val_mae={val_mae:.2f} | {sec:.1f}s{mark}")
        if bad >= cfg.PATIENCE:
            print(f"early stopping (best_epoch={best_epoch})")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    Path(cfg.CHECKPOINT_PATH).parent.mkdir(parents=True, exist_ok=True)
    torch.save({"model_state": model.state_dict(), "cfg": asdict(cfg), "val_mae": best_val}, cfg.CHECKPOINT_PATH)
    print("checkpoint:", cfg.CHECKPOINT_PATH)
    test_mae = evaluate(model, test_loader, device)
    print(f"DONE | val_mae={best_val:.2f} | test_mae={test_mae:.2f} | best_epoch={best_epoch}")
    return {
        "val_mae": best_val,
        "test_mae": test_mae,
        "best_epoch": best_epoch,
        "history": history,
        "model": model,
        "cfg": cfg,
        "device": device,
    }


def load_model_from_checkpoint(checkpoint_path: str, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    cfg = Config(**{k: v for k, v in ckpt["cfg"].items() if k in Config.__dataclass_fields__})
    model = MultimodalCalorieModel(cfg).to(device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    return model, cfg, ckpt


print("model / train / checkpoint helpers OK")


### Вывод этапа 2

Реализованы Dataset с путём `nutrition/images/<dish_id>/rgb.png`, аугментации/eval-transforms, мультимодальная модель с `head → 1`, AdamW с группами LR по ТЗ, `train(cfg)` с seed, L1Loss и сохранением чекпоинта.


## Этап 3. Обучение модели


In [ ]:
# @title 3.1. Конфиг, сплиты, loaders, train(cfg)
cfg = Config(
    NAME="full_c7_effb0",
    SEED=SEED,
    DATA_ROOT=str(DATA_ROOT),
    IMAGE_SIZE=160,
    BATCH_SIZE=4,  # ТЗ: небольшой batch
    EPOCHS=12,
    PATIENCE=3,
    HIDDEN_DIM=128,
    MAX_LENGTH=64,
    TEXT_MODEL="distilbert-base-uncased",
    IMAGE_MODEL="efficientnet_b0",
    TEXT_LR=2e-5,
    IMAGE_LR=2e-5,
    CLASSIFIER_LR=1e-3,
    USE_AUG=False,
    CHECKPOINT_PATH=CHECKPOINT_PATH,
    VAL_FRACTION=0.1,
)

work = dish[dish["exists"]].copy()
train_all = work[work["split"] == "train"].reset_index(drop=True)
test_df = work[work["split"] == "test"].reset_index(drop=True)
val_df = train_all.sample(frac=cfg.VAL_FRACTION, random_state=cfg.SEED)
train_df = train_all.drop(val_df.index).reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print("sizes train/val/test:", len(train_df), len(val_df), len(test_df))

tokenizer = AutoTokenizer.from_pretrained(cfg.TEXT_MODEL)
train_loader, val_loader, test_loader = make_loaders(
    train_df,
    val_df,
    test_df,
    tokenizer,
    image_size=cfg.IMAGE_SIZE,
    batch_size=cfg.BATCH_SIZE,
    use_aug=cfg.USE_AUG,
    max_length=cfg.MAX_LENGTH,
)

# sanity: один батч
b0 = next(iter(train_loader))
print("batch image", tuple(b0["image"].shape), "calories", tuple(b0["calories"].shape))

result = train(cfg, train_loader, val_loader, test_loader)


### Вывод этапа 3

Обучение запущено с `BATCH_SIZE=4`, `L1Loss`, head→1. Лучший чекпоинт сохраняется в `CHECKPOINT_PATH`. Итоговые MAE — в выводе ячейки выше (после прогона на GPU).


## Этап 4. Валидация качества (test MAE + топ-5 ошибок)


In [ ]:
# @title 4.1. Инференс с чекпоинта, test MAE
model, cfg_loaded, ckpt = load_model_from_checkpoint(CHECKPOINT_PATH, DEVICE)
tokenizer = AutoTokenizer.from_pretrained(cfg_loaded.TEXT_MODEL)

# пересобираем test loader на случай повторного запуска только этапа 4
work = dish[dish["exists"]].copy()
test_df = work[work["split"] == "test"].reset_index(drop=True)
test_ds = CaloriesDataset(
    test_df,
    get_eval_transforms(cfg_loaded.IMAGE_SIZE),
    tokenizer,
    cfg_loaded.MAX_LENGTH,
)
test_loader = DataLoader(
    test_ds,
    batch_size=cfg_loaded.BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_calories,
)

test_mae = evaluate(model, test_loader, DEVICE)
print(f"TEST MAE = {test_mae:.2f} kcal  |  target < 50: {test_mae < 50}")
print("checkpoint val_mae =", ckpt.get("val_mae"))


In [ ]:
# @title 4.2. Топ-5 ошибок: фото, ингредиенты, факт vs прогноз
model.eval()
rows = []
with torch.no_grad():
    for batch in test_loader:
        pred = model(
            batch["image"].to(DEVICE),
            batch["input_ids"].to(DEVICE),
            batch["attention_mask"].to(DEVICE),
        ).cpu().numpy()
        y = batch["calories"].numpy()
        for i in range(len(y)):
            rows.append(
                {
                    "dish_id": batch["dish_id"][i],
                    "text": batch["text"][i],
                    "y_true": float(y[i]),
                    "y_pred": float(pred[i]),
                    "abs_err": float(abs(pred[i] - y[i])),
                    "mass": float(batch["mass"][i]),
                }
            )

err_df = pd.DataFrame(rows).sort_values("abs_err", ascending=False)
top5 = err_df.head(5).reset_index(drop=True)
print(top5[["dish_id", "y_true", "y_pred", "abs_err", "mass"]].to_string(index=False))

id2path = {str(k): v for k, v in dish.set_index("dish_id")["image_path"].to_dict().items()}
fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
for ax, (_, r) in zip(axes, top5.iterrows()):
    path = id2path[str(r["dish_id"])]
    ax.imshow(Image.open(path).convert("RGB"))
    ax.set_title(
        f"true {r['y_true']:.0f} / pred {r['y_pred']:.0f}\nerr {r['abs_err']:.0f}\n{r['text'][:36]}…",
        fontsize=7,
    )
    ax.axis("off")
plt.suptitle("Топ-5 ошибок на test")
plt.tight_layout()
plt.show()

for i, r in top5.iterrows():
    print(f"\n#{i+1} {r['dish_id']}")
    print(f"  ingredients: {r['text']}")
    print(
        f"  true={r['y_true']:.1f}  pred={r['y_pred']:.1f}  "
        f"|err|={r['abs_err']:.1f}  mass={r['mass']:.1f}"
    )


### Вывод этапа 4 — разбор ошибок

Типичные причины больших |err| на test:

1. **Нетипичная масса порции** при похожем наборе ингредиентов на фото — модель опирается на вид тарелки и список продуктов, но калории сильно зависят от граммовки.
2. **Редкие / смешанные ингредиенты** — мало примеров в train, текстовые токены редкие.
3. **Визуально похожие блюда с разной калорийностью** (соусы, масло, скрытые добавки не видны на rgb).
4. **Качество кадра** (ракурс, обрезка, освещение) ухудшает image-ветку.

Если test MAE ≥ 50 — эти примеры задают гипотезы для доработки (учёт `total_mass` как признака, сильнее текст, балансировка по калориям).
